In [4]:
# Stage 1: 双解码器VAE训练
# 使用Dual Decoder架构 + MMD对齐SC和ST模态
%run stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_SC.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad" \
    --n_epochs 50 \
    --resolution 3 \
    --loss_type zinb \
    --beta 0.1 \
    --lambda_mmd 1.0 \
    --top_n_per_type 200 \
    --latent_dim 256 \
    --output_dir ./stage1_results/PDAC 

# \
#     --precomputed_marker_file "/mnt/d/ST_Graduation_Project/SC_MAP_ST/stage1_results/PDAC/final_genes.txt"

Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 200
   Leiden resolution: 3.0
   Batch size: 256
   Epochs: 50
   Learning rate: 0.001
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project/database/PDAC/PDAC_SC.h5ad
   SC shape: (1926, 19738)
   Loading ST: /mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad
   ST shape: (428, 19738)
   Common genes: 19738
   SC final: (1926, 19738)
   ST final: (428, 19738)
Computing clusters and marker genes...
Starting clustering analysis...
   SC shape: (1926, 19738)
   Loading ST: /mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad
   ST shape: (428, 19738)
   Common genes: 19738
   SC final: (1926, 19738)
   ST final: (428, 19738)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 35 clusters
Clustering r

VAE Training: 100%|██████████| 50/50 [01:06<00:00,  1.32s/epoch, Train=1666.1133, Recon=1660.8071, KL=53.0601, MMD=0.0302, Test=1733.6016]



Computing cluster centers...
   SC training data: (1733, 1288)
   Number of clusters: 35
   Computing full gene cluster expressions...
      Total genes: 19738
   Completed: 35 clusters with center and expressions (all genes)
Visualizing modality alignment...
Generating UMAP visualization for modality alignment...
   Computing UMAP on 2118 samples with 256 dims...
   UMAP visualization saved to: ./stage1_results/PDAC/modality_alignment_umap.png
Saving model to: ./stage1_results/PDAC/final_vae.pth
   Cluster centers: 35 clusters
   Cluster expressions (marker genes): 35 clusters
   Cluster expressions (all genes): 35 clusters
   Cluster expressions (all genes, count backup): 35 clusters
   Saved successfully!
   UMAP visualization saved to: ./stage1_results/PDAC/modality_alignment_umap.png
Saving model to: ./stage1_results/PDAC/final_vae.pth
   Cluster centers: 35 clusters
   Cluster expressions (marker genes): 35 clusters
   Cluster expressions (all genes): 35 clusters
   Cluster expre

---

## Stage 2 参数说明

**自动计算 `cells_per_spot`** (scale 因子):
- 如果不指定 `--cells_per_spot`，会自动计算：
  - 公式: `mean(spot_total_counts) / mean(cell_total_counts)` 
  - ⚠️ **使用 raw counts**（优先级：`n_counts` > `raw` layer > 当前 X）
  - **这是一个 scale 因子**，反映测序深度差异，不一定是物理细胞数
  - 可能 < 1（SC 测序更深）或 > 10（ST 测序更深）
  - 只在 > 30 或 < 0.1 时才会截断并警告

**图结构参数**:
- `k_spatial`: 每个 spot 连接多少个邻居 spot（空间平滑）
- `k_celltype`: 每个 spot 连接多少个 celltype（反卷积灵活性）
  - ⚠️ 如果 > 实际簇数，会自动截断（会有警告）
  - 建议: ≤ 簇数的 50-80% 以保持稀疏性

In [ ]:
# 测试自动计算 cells_per_spot (scale 因子)
import scanpy as sc
import os

st_file = "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad"
stage1_dir = "./stage1_results/PDAC/"
sc_clustered_file = os.path.join(stage1_dir, 'sc_adata_clustered.h5ad')

if os.path.exists(st_file) and os.path.exists(sc_clustered_file):
    # Load ST data
    st_adata = sc.read_h5ad(st_file)
    
    # Get average spot counts (prefer n_counts, then raw, then X)
    if 'n_counts' in st_adata.obs.columns:
        avg_spot_counts = st_adata.obs['n_counts'].mean()
        print(f"ST: Using n_counts")
    elif hasattr(st_adata, 'raw') and st_adata.raw is not None:
        avg_spot_counts = st_adata.raw.X.sum(axis=1).mean()
        print(f"ST: Using raw layer")
    else:
        avg_spot_counts = st_adata.X.sum(axis=1).mean()
        print(f"ST: Using X (may be normalized!)")
    
    # Load SC data
    sc_adata = sc.read_h5ad(sc_clustered_file)
    
    # Get average cell counts
    if 'n_counts' in sc_adata.obs.columns:
        avg_cell_counts = sc_adata.obs['n_counts'].mean()
        print(f"SC: Using n_counts")
    elif hasattr(sc_adata, 'raw') and sc_adata.raw is not None:
        avg_cell_counts = sc_adata.raw.X.sum(axis=1).mean()
        print(f"SC: Using raw layer")
    else:
        avg_cell_counts = sc_adata.X.sum(axis=1).mean()
        print(f"SC: Using X (may be normalized!)")
    
    # Calculate scale factor
    scale_factor = avg_spot_counts / avg_cell_counts
    
    print(f"\n平均 spot 总 count: {avg_spot_counts:.1f}")
    print(f"平均 cell 总 count: {avg_cell_counts:.1f}")
    print(f"Scale 因子 (cells_per_spot): {scale_factor:.3f}")
    
    if scale_factor < 1:
        print(f"→ SC 测序深度更高 ({1/scale_factor:.1f}x)")
    else:
        print(f"→ ST 测序深度更高 ({scale_factor:.1f}x)")
else:
    print("数据文件不存在")

In [3]:
# k_celltype 影响比较大
# cells_per_spot 会自动计算（如果需要手动指定，添加 --cells_per_spot 10）
%run stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad" \
    --stage1_model_path "./stage1_results/PDAC/final_vae.pth" \
    --output_dir "./stage2_results/PDAC/" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 1e-4 \
    --loss_lambda_mse 0.1 \
    --loss_lambda_cosine 5 \
    --loss_lambda_pearson 2 \
    --loss_lambda_reg 0.1 \
    --loss_lambda_sparse 0.01 \
    --loss_lambda_proportion 2 \
    --k_spatial 10 \
    --k_celltype 30 \
    --batch_size 512 \
    --n_epochs 500 \
    --weight_threshold 0.01

Sample name: PDAC
Stage 1 model: ./stage1_results/PDAC/final_vae.pth
Output directory: ./stage2_results/PDAC/
Device: cpu
Weight threshold: 0.01
Loading pretrained VAE Encoder...
   VAE architecture: 1300 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 1926 cell cluster labels from checkpoint
Loaded cluster centers: torch.Size([35, 256])
Loaded cluster expressions: torch.Size([35, 1300])
Loaded full gene expressions (count): 35 clusters × 19738 genes
Loaded all genes list: 19738 genes
VAE Encoder loaded: 1300 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '4', '5', '6', '7', '8', '9']
Marker genes: 1300
Using Stage 1 cluster centers and expressions...
Loaded 35 clusters
Using Stage 1 pretrained cluster data
   Cluster centers: torch.Size([35, 256])
   Cluster expressions: torch.Size([35, 1300])
Load

GAT Training:   3%|▎         | 17/500 [00:15<07:04,  1.14epoch/s, Total=99.1941, Pearson=0.5811, MSE=951.6549, Cosine=0.4595, Proportion=0.2840] Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7a3a091742b0>>
Traceback (most recent call last):
  File "/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7a3a091742b0>>
Traceback (most recent call last):
  File "/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
GAT Training:  17%|█▋        | 83/500 [01:25<07:08,  1.03s/epoch, Total=83.8164, Pearson=0.5635, MSE=795.0640, Cosine=0.4468,

KeyboardInterrupt: 